# Read a .npy file in Python with chDB

Companion to [How to read an NPY file in Python](https://clickhouse.com/resources/engineering/read-npy-file-python).

Run `./generate.sh` first to create `data/`. Requires `pip install chdb pandas numpy`.

## 1. Read a .npy file into a DataFrame

In [ ]:
from chdb.datastore import DataStore

df = DataStore.from_file("data/readings.npy", format="Npy")
df

In [ ]:
df.dtypes

## 2. Filter and aggregate the way you already do

In [ ]:
print(f"mean: {float(df['array'].mean()):.2f}")
print(f"max: {float(df['array'].max()):.2f}  min: {float(df['array'].min()):.2f}")

## 3. A 2-D .npy reads as one row per outer element

In [ ]:
mat = DataStore.from_file("data/matrix.npy", format="Npy")
mat

## 4. Join two arrays by position (readings + quality flags)

In [ ]:
readings_pdf = df.to_pandas().rename(columns={"array": "reading"})
flags_pdf = DataStore.from_file("data/flags.npy", format="Npy").to_pandas().rename(columns={"array": "ok"})
combined = readings_pdf.join(flags_pdf)
combined

## 5. Hand off to real pandas when you need it

In [ ]:
pdf = df.to_pandas()   # a real pandas.DataFrame, in memory
type(pdf)

In [ ]:
pdf[pdf["array"] > 75]

## 6. Performance: DataStore vs numpy.load (mean of 3M-element array)

NumPy wins the raw vectorized reduction. chDB adds value when you need the result in DataFrame form or when you are combining array data with other file formats.

Apple M4 Pro (14 cores, 24 GB RAM, macOS); chDB 4.1.8, Python 3.14; best-of-3, warm.

In [ ]:
import time
import numpy as np

def datastore_mean():
    d = DataStore.from_file("data/large.npy", format="Npy")
    return float(d["array"].mean())

def numpy_mean():
    v = np.load("data/large.npy")
    return v.mean()

def best_of_3(fn):
    fn()  # warm
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

np_s = best_of_3(numpy_mean)
ds_s = best_of_3(datastore_mean)
print(f"numpy.load + .mean():           {np_s:.3f}s")
print(f"DataStore.from_file + .mean():  {ds_s:.3f}s")
if ds_s < np_s:
    print(f"speedup:                        {np_s / ds_s:.1f}x")
else:
    print(f"numpy faster by:                {ds_s / np_s:.1f}x")